In [1]:
from jointfm_client import bootstrap_notebook
bootstrap_notebook(add_src_root=True)

Changed working directory to: /home/stefan/workspace/joint-client-python


PosixPath('/home/stefan/workspace/joint-client-python')

# Mean Forecast
Forecast USD portfolio NAV and risk from 100 positive float daily observations: equity index level, 10-year Treasury yield, and one EUR/USD FX rate. Request mean forecasts for portfolio NAV and realized volatility over the next 10 ordinal horizons.

In [2]:
from pathlib import Path

import pandas as pd

from jointfm_client import ColumnSpec, JointFMClient

HISTORY_PATH = Path('notebooks/history.csv')
FEATURE_COLUMNS = ['equity_index_level', 'treasury_10y_yield', 'eur_usd_rate']
TARGET_COLUMNS = ['portfolio_nav', 'realized_volatility']
INPUT_STEPS = 100
OUTPUT_HORIZONS = 10
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(f'Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}')
if len(history) != INPUT_STEPS:
    raise ValueError(f'Expected {INPUT_STEPS} history rows, got {len(history)}')

columns = [
    *[
        ColumnSpec(name=column_name, modality='numeric', role='feature')
        for column_name in FEATURE_COLUMNS
    ],
    *[
        ColumnSpec(name=column_name, modality='numeric', role='target')
        for column_name in TARGET_COLUMNS
    ],
]
client = JointFMClient.from_env()
result = client.forecast_mean(
    history,
    query_times=QUERY_TIMES,
    requested_columns=TARGET_COLUMNS,
    columns=columns,
    seed=7,
)
forecast = result.to_pandas_tidy()
expected_forecast_rows = OUTPUT_HORIZONS * len(TARGET_COLUMNS)
if len(forecast) != expected_forecast_rows:
    raise ValueError(f'Expected {expected_forecast_rows} forecast rows, got {len(forecast)}')
forecast

,query_time,requested_column,value
0,100,portfolio_nav,112.878296
1,100,realized_volatility,0.011236
2,101,portfolio_nav,113.012344
3,101,realized_volatility,0.011282
4,102,portfolio_nav,112.916306
5,102,realized_volatility,0.011246
6,103,portfolio_nav,113.054443
7,103,realized_volatility,0.011297
8,104,portfolio_nav,112.993118
9,104,realized_volatility,0.011270
